In [ ]:
!pip install torch torchvision torchaudio

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

GPU available: True
Device name: Tesla T4


In [ ]:
pip install youtube-transcript-api

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

ytt_api = YouTubeTranscriptApi()
ytt_api.fetch('YpI0jgqNJGc')

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='[Music]', start=2.04, duration=16.41), FetchedTranscriptSnippet(text='dad', start=16.4, duration=6.709), FetchedTranscriptSnippet(text='[Music]', start=18.45, duration=4.659), FetchedTranscriptSnippet(text="Bing it's", start=26.679, duration=7.521), FetchedTranscriptSnippet(text="hot oh it's so hot mom what are we doing", start=28.8, duration=8.8), FetchedTranscriptSnippet(text="today nothing until you've cleaned your", start=34.2, duration=6.8), FetchedTranscriptSnippet(text="teeth but I don't want to clean my teeth", start=37.6, duration=7.68), FetchedTranscriptSnippet(text="it's boring boring things are still", start=41.0, duration=8.199), FetchedTranscriptSnippet(text="important no they're not you sound like", start=45.28, duration=4.959), FetchedTranscriptSnippet(text='your', start=49.199, duration=3.84), FetchedTranscriptSnippet(text='dad hey squirts Uncle strip said we', start=50.239, duration=5.96), FetchedTranscriptSni

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
ytt_api = YouTubeTranscriptApi()
FetchedTranscript = ytt_api.fetch('YpI0jgqNJGc')
TranscriptString = ' '.join(snippet.text for snippet in FetchedTranscript.snippets)

In [ ]:
print(TranscriptString)

[Music] dad [Music] Bing it's hot oh it's so hot mom what are we doing today nothing until you've cleaned your teeth but I don't want to clean my teeth it's boring boring things are still important no they're not you sound like your dad hey squirts Uncle strip said we could use his pool while he's in Bary yeah let's get out of here make sure you take all the swimming stuff yeah yeah we got it covered don't just get the fun stuff I mean the bag of stuff boring yeah boring hey it's nice and cool actually this episode of Lou is called the pool mom is such a fast pot isn't she she is mom always makes us do so many boring things she he does that is way more fun I am dad what does [Music] a k k k k k k j j k k k [Music] spell well stick your thongs on I didn't bring them ah okay um how are we going to do this ow ow ow hey Bingo what's going on up there I'm a gir and I'm eating these leaves and someone Open this gate I have my rashy did you bring it no then no dad did we bring some Sun scream

In [ ]:
pip install transformers torch sentencepiece

In [ ]:
pip install tf-keras

In [ ]:
from transformers import pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cuda:0


In [ ]:
#Q1
summary = summarizer(
    TranscriptString,
    max_length=300,
    min_length=50,
    do_sample=False
)
summary_text = summary[0]['summary_text']
print(summary_text)


Lou and his family use his uncle's pool while he's in Bary. The children play in the pool until they get bored. They also use the pool to learn how to shake properly. The kids also use it to practice their swimming skills.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("facebook/opt-1.3b")
model = AutoModelForCausalLM.from_pretrained("facebook/opt-1.3b")

prompt = "generate a conversation between the characters bandit and bluey from the tv show Bluey"

inputs = tokenizer(prompt, return_tensors="pt")
output = model.generate(
    **inputs,
    max_length=100,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.7
)
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

generate a conversation between the characters bandit and bluey from the tv show Bluey.

This is a game where you can use the chat and the chat box.

If you have trouble, please contact me.


In [ ]:
pip install transformers datasets accelerate

In [ ]:
!pip install pyarrow

In [ ]:
import pyarrow.parquet as pq

In [ ]:
import os
print(os.path.exists("/bluey_script.txt"))

True


In [ ]:
# 0. Install required packages
!pip install transformers datasets accelerate --quiet

# 1. Imports
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset
import torch
import os

# 2. Set memory config to reduce fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 3. Load dataset from uploaded file
dataset = load_dataset("text", data_files={"train": "/bluey_script.txt"})

# 4. Load tokenizer and model (smaller model for Colab GPU)
model_name = "facebook/opt-350m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 5. Tokenize dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# 6. Data collator for causal LM
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# 7. Training arguments (optimized for Colab)
training_args = TrainingArguments(
    output_dir="./opt-350m-bluey",
    overwrite_output_dir=True,
    per_device_train_batch_size=1,
    num_train_epochs=1,  # reduce for speed
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

# 8. Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

# 9. Train the model
trainer.train()

# 10. Save the fine-tuned model and tokenizer
trainer.save_model("./opt-350m-bluey")
tokenizer.save_pretrained("./opt-350m-bluey")

/tmp/ipython-input-350481977.py:53: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
50,2.222000
100,1.623300
150,1.340400
200,1.689600
250,1.153800
300,1.325600
350,1.000000


('./opt-350m-bluey/tokenizer_config.json',
 './opt-350m-bluey/special_tokens_map.json',
 './opt-350m-bluey/vocab.json',
 './opt-350m-bluey/merges.txt',
 './opt-350m-bluey/added_tokens.json',
 './opt-350m-bluey/tokenizer.json')

In [ ]:
trainer.save_model("./opt-350m-bluey")
tokenizer.save_pretrained("./opt-350m-bluey")
generator = pipeline("text-generation", model="./opt-350m-bluey", tokenizer="./opt-350m-bluey", device=0)

Device set to use cuda:0


In [ ]:
#Q3
from transformers import pipeline

generator = pipeline("text-generation", model="./opt-350m-bluey", tokenizer="./opt-350m-bluey", device=0)

prompt = "generate a conversation between the characters bandit and bluey from the tv show Bluey"
output = generator(prompt, max_length=100, do_sample=True, temperature=0.7)

print(output[0]["generated_text"])

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=101) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


generate a conversation between the characters bandit and bluey from the tv show Bluey: You're trying to get us down! We can't take out all the swim stuff! I'm not. Hey, Bingo! Dad! Did you bring any food? Boring. Dad! You're still not. You don't wanna be! She's boring! She didn't bring any food! Okay! We'll just have to stay in the pool! Yeah, that's fun! What's going on do you think I can do? You'll be able to stay in the pool! Dad! You'll be able to get us all the food we'd need! You'll be able to get us all the way! Okay, didn't bring 'em! I think we can't take this out! We'll have to have to get up and swim, sure! Yeah! Okay, I'll have to get up here! Oh, yeah! We'll just have to bring them! Yeah! Okay! Okay, we'll just have to swim! Dad! Okay, yeah! Yeah, we'll just have to be sitting here! Yeah, we'll just have to swim! Yeah! Yeah, you won't be able to! Oh, yeah! Okay! Yeah! Yeah! Yeah! Okay! Yeah! Yeah! You'll have to stay
